In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/ai4i2020.csv")

df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [2]:
df_model = df.drop(columns=["UDI", "Product ID"])
df_model = pd.get_dummies(df_model, columns=["Type"])

df_model.columns = [
    col.replace("[","")
       .replace("]","")
       .replace("<","")
       .replace(" ","_")
       .replace("/","_")
    for col in df_model.columns
]

In [3]:
df_model.columns

Index(['Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm',
       'Torque_Nm', 'Tool_wear_min', 'Machine_failure', 'TWF', 'HDF', 'PWF',
       'OSF', 'RNF', 'Type_H', 'Type_L', 'Type_M'],
      dtype='str')

In [4]:
df_model["temperature_difference"] = (
    df_model["Process_temperature_K"] -
    df_model["Air_temperature_K"]
)

In [5]:
df_model["mechanical_stress"] = (
    df_model["Torque_Nm"] *
    df_model["Rotational_speed_rpm"]
)

In [6]:
df_model["wear_stress"] = (
    df_model["Tool_wear_min"] *
    df_model["Torque_Nm"]
)

In [7]:
df_model["thermal_wear_stress"] = (
    df_model["Process_temperature_K"] *
    df_model["Tool_wear_min"]
)

In [8]:
df_model["power_load"] = (
    df_model["Torque_Nm"] *
    df_model["Rotational_speed_rpm"] / 1000
)

In [9]:
df_model.tail()

,Air_temperature_K,Process_temperature_K,Rotational_speed_rpm,Torque_Nm,Tool_wear_min,Machine_failure,TWF,HDF,PWF,OSF,RNF,Type_H,Type_L,Type_M,temperature_difference,mechanical_stress,wear_stress,thermal_wear_stress,power_load
9995,298.8,308.4,1604,29.5,14,0,0,0,0,0,0,False,False,True,9.6,47318.0,413.0,4317.6,47.3180
9996,298.9,308.4,1632,31.8,17,0,0,0,0,0,0,True,False,False,9.5,51897.6,540.6,5242.8,51.8976
9997,299.0,308.6,1645,33.4,22,0,0,0,0,0,0,False,False,True,9.6,54943.0,734.8,6789.2,54.9430
9998,299.0,308.7,1408,48.5,25,0,0,0,0,0,0,True,False,False,9.7,68288.0,1212.5,7717.5,68.2880
9999,299.0,308.7,1500,40.2,30,0,0,0,0,0,0,False,False,True,9.7,60300.0,1206.0,9261.0,60.3000


In [10]:
X = df_model.drop(columns=["Machine_failure"])
y = df_model["Machine_failure"]

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train.values, y_train.values)

y_pred = xgb_model.predict(X_test.values)
y_prob = xgb_model.predict_proba(X_test.values)[:,1]

In [13]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1932
           1       1.00      0.94      0.97        68

    accuracy                           1.00      2000
   macro avg       1.00      0.97      0.98      2000
weighted avg       1.00      1.00      1.00      2000

ROC-AUC: 0.9975642430885399


In [14]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(xgb_model, "../models/failure_model_v2.pkl")

['../models/failure_model_v2.pkl']